# CosyVoice3 Fine-Tuning (Debi & Marlene) - Colab A100

Base Model: `FunAudioLLM/Fun-CosyVoice3-0.5B-2512`

- LLM SFT only (Flow/HiFiGAN pretrained 유지)
- Multi-speaker: debi + marlene
- 4 style tags: `[excited]`, `[sad]`, `[happy]`, `[calm]`
- Colab 시스템 Python + PyTorch 그대로 사용 (conda 없음)

공식 파이프라인 (`examples/libritts/cosyvoice3/run.sh`) 기반

---
## 1. GPU 확인 + Drive 마운트

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi

import torch, os
print(f'\ntorch {torch.__version__} CUDA={torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

WORK_DIR = '/content/CosyVoice'
DATA_DIR = '/content/cosyvoice3_data'
BACKUP_DIR = '/content/drive/MyDrive/cosyvoice3_backup'
os.makedirs(BACKUP_DIR, exist_ok=True)

Mounted at /content/drive
Sat Mar  7 06:08:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------

---
## 2. 데이터 업로드 (Drive zip)

In [2]:
import os, shutil

DATA_DIR = '/content/cosyvoice3_data'
DRIVE_ZIP = '/content/drive/MyDrive/cosyvoice3_data.zip'

if os.path.exists(DRIVE_ZIP):
    if os.path.exists(DATA_DIR):
        shutil.rmtree(DATA_DIR)
    !unzip -q "{DRIVE_ZIP}" -d /content/
    print(f'Unzipped: {DRIVE_ZIP}')
elif not os.path.exists(DATA_DIR):
    from google.colab import files
    uploaded = files.upload()
    !unzip -q cosyvoice3_data.zip -d /content/
else:
    print(f'Data exists: {DATA_DIR}')

for split in ['train', 'dev']:
    d = os.path.join(DATA_DIR, split)
    if os.path.exists(d):
        wc = len([f for f in os.listdir(os.path.join(d,'wavs')) if f.endswith('.wav')])
        print(f'{split}: {wc} wavs')

Unzipped: /content/drive/MyDrive/cosyvoice3_data.zip
train: 387 wavs
dev: 42 wavs


---
## 3. CosyVoice Clone + 의존성 설치

Colab 시스템 Python/PyTorch를 그대로 사용합니다.
CosyVoice `requirements.txt`에서 이미 있거나 불필요한 것만 제외하고 설치합니다.

In [3]:
%%bash
set -e

# === CosyVoice clone ===
cd /content
if [ ! -d "CosyVoice" ]; then
    git clone --depth 1 https://github.com/FunAudioLLM/CosyVoice.git
    cd CosyVoice && git submodule update --init --recursive
else
    echo "CosyVoice already cloned"
fi

# === path.sh ===
cd /content/CosyVoice
mkdir -p examples/libritts/cosyvoice3
cat > examples/libritts/cosyvoice3/path.sh << 'EOF'
export PYTHONPATH=/content/CosyVoice/third_party/Matcha-TTS:/content/CosyVoice:${PYTHONPATH:-}
EOF

echo "Done: $(ls -d /content/CosyVoice)"

Submodule path 'third_party/Matcha-TTS': checked out 'dd9105b34bf2be2230f4aa1e4769fb586a3c824e'
Done: /content/CosyVoice


Cloning into 'CosyVoice'...
Submodule 'third_party/Matcha-TTS' (https://github.com/shivammehta25/Matcha-TTS.git) registered for path 'third_party/Matcha-TTS'
Cloning into '/content/CosyVoice/third_party/Matcha-TTS'...


In [4]:
%%bash
set -e
cd /content/CosyVoice

echo "=== [1/6] 기본 도구 ==="
pip install setuptools wget

echo "=== [2/6] ML 핵심 ==="
pip install onnxruntime-gpu conformer hyperpyyaml "protobuf>=5.26,<6"

echo "=== [3/6] 학습 프레임워크 ==="
pip install lightning accelerate diffusers transformers

echo "=== [4/6] 오디오 + 유틸 ==="
pip install soundfile librosa inflect pyarrow pydantic tiktoken more-itertools huggingface_hub

echo "=== [5/6] whisper + numba 호환 ==="
pip install --upgrade numba
pip install openai-whisper --no-deps

echo "=== [6/6] Matcha-TTS (Cython 빌드만) ==="
cd third_party/Matcha-TTS
pip install cython
python setup.py build_ext --inplace

echo ""
echo "=== All Done ==="

=== [1/6] 기본 도구 ===
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9655 sha256=ca98a06de3355d93b9ce8db853eaa1680ff06f2a44973c921da51587f4b9e795
  Stored in directory: /root/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built wget
=== [2/6] ML 핵심 ===
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.8/252.8 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 62.3 MB/s eta 0:00:00
=== [3/6] 학습 프레임워크 ===
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 82.0 MB/s eta 0:00:

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.64.0 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.64.0 which is incompatible.
performance hint: matcha/utils/monotonic_align/core.pyx:11:5: Exception check on 'maximum_path_each' will always require the GIL to be acquired.
Possible solutions:
	1. Declare 'maximum_path_each' as 'noexcept' if you control the definition and you're sure you don't want the function to raise exceptions.
	2. Use an 'int' return type on 'maximum_path_each' to allow an error code to be returned.
performance hint: matcha/utils/monotonic_align/core.pyx:42:6: Exception check on 'maximum_path_c' will always require the GIL to be acquired.
Possible solutions:
	1. Declare 'maximum_path_c' as 'noexcept' if you control the definiti

In [6]:
!pip install pyworld

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.0/261.0 kB 7.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyworld: filename=pyworld-0.3.5-cp312-cp312-linux_x86_64.whl size=943061 sha256=bb178784c9e5e4ea8172d7c07d1b03f8786dd9901ef059910d11d97f3f43d5bd
  Stored in directory: /root/.cache/pip/wheels/be/ac/58/c6a1791ec6d17f3a99b6ccdec92b472f560cb5c564b83dd77e
Successfully built pyworld


In [7]:
# 검증
import sys
sys.path.insert(0, '/content/CosyVoice/third_party/Matcha-TTS')
sys.path.insert(0, '/content/CosyVoice')

import torch; print(f'torch {torch.__version__} CUDA={torch.cuda.is_available()}')
import numpy as np; print(f'numpy {np.__version__}')
import pkg_resources; print('pkg_resources OK')
import onnxruntime as ort; print(f'onnxruntime {ort.__version__}')
import whisper; print('whisper OK')
import transformers; print(f'transformers {transformers.__version__}')
from hyperpyyaml import load_hyperpyyaml; print('HyperPyYAML OK')
import matcha; print('matcha OK')
from cosyvoice.dataset.processor import parquet_opener; print('cosyvoice.dataset OK')
print('\n=== All OK ===')

torch 2.10.0+cu128 CUDA=True
numpy 2.0.2
pkg_resources OK
onnxruntime 1.24.3
whisper OK
transformers 5.0.0
HyperPyYAML OK
matcha OK
cosyvoice.dataset OK

=== All OK ===


---
## 4. Pretrained 모델 다운로드

In [31]:
!rm -rf /content/CosyVoice/pretrained_models/Fun-CosyVoice3-0.5B

In [32]:
import os
from huggingface_hub import snapshot_download

MODEL_DIR = '/content/CosyVoice/pretrained_models/Fun-CosyVoice3-0.5B'

if not os.path.exists(MODEL_DIR) or not os.listdir(MODEL_DIR):
    os.makedirs(os.path.dirname(MODEL_DIR), exist_ok=True)
    snapshot_download(
        repo_id='FunAudioLLM/Fun-CosyVoice3-0.5B-2512',
        local_dir=MODEL_DIR,
        ignore_patterns=['*.md'],
    )
    print('Downloaded')
else:
    print(f'Exists: {MODEL_DIR}')

for f in ['llm.pt', 'flow.pt', 'hift.pt', 'campplus.onnx', 'speech_tokenizer_v3.onnx', 'cosyvoice3.yaml']:
    p = os.path.join(MODEL_DIR, f)
    s = f'{os.path.getsize(p)/1e6:.1f}MB' if os.path.exists(p) else 'MISSING'
    print(f'  {f}: {s}')
print(f'  CosyVoice-BlankEN/: {"OK" if os.path.isdir(os.path.join(MODEL_DIR, "CosyVoice-BlankEN")) else "MISSING"}')

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

Downloaded
  llm.pt: 2024.7MB
  flow.pt: 1329.1MB
  hift.pt: 83.2MB
  campplus.onnx: 28.3MB
  speech_tokenizer_v3.onnx: 969.5MB
  cosyvoice3.yaml: 0.0MB
  CosyVoice-BlankEN/: OK


---
## 5. wav.scp 경로 변환 + 데이터 링크

In [23]:
import os

DATA_DIR = '/content/cosyvoice3_data'

for split in ['train', 'dev']:
    split_dir = os.path.join(DATA_DIR, split)
    scp_path = os.path.join(split_dir, 'wav.scp')

    with open(scp_path) as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        parts = line.strip().split(' ', 1)
        if len(parts) != 2: continue
        utt_id, wav_path = parts
        if not wav_path.startswith('/'):
            wav_path = os.path.join(split_dir, wav_path)
        if '\\' in wav_path or (':' in wav_path and wav_path[1] == ':'):
            wav_path = os.path.join(split_dir, 'wavs', os.path.basename(wav_path))
        new_lines.append(f'{utt_id} {wav_path}\n')

    with open(scp_path, 'w') as f:
        f.writelines(new_lines)

    sample = new_lines[0].strip().split(' ', 1)[1]
    print(f'{split}: {len(new_lines)} entries, exists={os.path.exists(sample)}')

train: 387 entries, exists=True
dev: 42 entries, exists=True


In [24]:
%%bash
cd /content/CosyVoice/examples/libritts/cosyvoice3

mkdir -p data
ln -sfn /content/cosyvoice3_data/train data/train
ln -sfn /content/cosyvoice3_data/dev data/dev

wc -l data/train/wav.scp data/train/text data/dev/wav.scp data/dev/text
echo ''
head -1 data/train/text

  387 data/train/wav.scp
  387 data/train/text
   42 data/dev/wav.scp
   42 data/dev/text
  858 total

debi_taunt_2_01_01 You are a helpful assistant.<|endofprompt|>[excited]셋줄게, 도망쳐봐


---
## 6. Speaker Embedding 추출 (CAMPlus)

In [25]:
%%bash
cd /content/CosyVoice/examples/libritts/cosyvoice3
. ./path.sh

PRETRAINED="../../../pretrained_models/Fun-CosyVoice3-0.5B"

for split in train dev; do
    echo "=== Embedding: $split ==="
    python ../../../tools/extract_embedding.py \
        --dir data/$split \
        --onnx_path $PRETRAINED/campplus.onnx \
        --num_thread 4
done

for split in train dev; do
    ls -lh data/$split/utt2embedding.pt data/$split/spk2embedding.pt 2>/dev/null || echo "$split: MISSING"
done

=== Embedding: train ===
=== Embedding: dev ===
-rw-r--r-- 1 root root 4.7K Mar  7 06:25 data/train/spk2embedding.pt
-rw-r--r-- 1 root root 670K Mar  7 06:25 data/train/utt2embedding.pt
-rw-r--r-- 1 root root 4.7K Mar  7 06:25 data/dev/spk2embedding.pt
-rw-r--r-- 1 root root  74K Mar  7 06:25 data/dev/utt2embedding.pt


387it [00:06, 61.71it/s]
42it [00:01, 23.63it/s]


---
## 7. Speech Token 추출 (speech_tokenizer_v3)

In [26]:
%%bash
cd /content/CosyVoice/examples/libritts/cosyvoice3
. ./path.sh

PRETRAINED="../../../pretrained_models/Fun-CosyVoice3-0.5B"

for split in train dev; do
    echo "=== Speech Token: $split ==="
    python ../../../tools/extract_speech_token.py \
        --dir data/$split \
        --onnx_path $PRETRAINED/speech_tokenizer_v3.onnx
done

for split in train dev; do
    ls -lh data/$split/utt2speech_token.pt 2>/dev/null || echo "$split: MISSING"
done

=== Speech Token: train ===
=== Speech Token: dev ===
-rw-r--r-- 1 root root 72K Mar  7 06:27 data/train/utt2speech_token.pt
-rw-r--r-- 1 root root 8.5K Mar  7 06:27 data/dev/utt2speech_token.pt


2026-03-07 06:26:10.456569441 [W:onnxruntime:, transformer_memcpy.cc:111 ApplyImpl] 14 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
2026-03-07 06:26:10.465295687 [W:onnxruntime:, session_state.cc:1327 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2026-03-07 06:26:10.465314421 [W:onnxruntime:, session_state.cc:1329 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.
0it [00:00, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec

---
## 8. Parquet 변환

In [27]:
%%bash
cd /content/CosyVoice/examples/libritts/cosyvoice3
. ./path.sh

for split in train dev; do
    echo "=== Parquet: $split ==="
    mkdir -p data/$split/parquet
    python ../../../tools/make_parquet_list.py \
        --num_utts_per_parquet 1000 \
        --num_processes 4 \
        --src_dir data/$split \
        --des_dir data/$split/parquet
done

cat data/train/parquet/data.list > data/train.data.list
cat data/dev/parquet/data.list > data/dev.data.list

echo ''
echo 'train.data.list:'
cat data/train.data.list
echo 'dev.data.list:'
cat data/dev.data.list

=== Parquet: train ===
=== Parquet: dev ===

train.data.list:
data/train/parquet/parquet_000000000.tar
dev.data.list:
data/dev/parquet/parquet_000000000.tar


100%|██████████| 42/42 [00:00<00:00, 12050.12it/s]


---
## 9. Config 준비 + deepspeed 패치

In [28]:
import os, shutil, re

TRAIN_BASE = '/content/CosyVoice/examples/libritts/cosyvoice3'
CONF_DIR = os.path.join(TRAIN_BASE, 'conf')
os.makedirs(CONF_DIR, exist_ok=True)

# cosyvoice3.yaml
cfg_dst = os.path.join(CONF_DIR, 'cosyvoice3.yaml')
cfg_src = '/content/CosyVoice/pretrained_models/Fun-CosyVoice3-0.5B/cosyvoice3.yaml'
if not os.path.exists(cfg_dst):
    shutil.copy2(cfg_src, cfg_dst)
    print(f'Config copied')

with open(cfg_dst) as f:
    for line in f:
        if 'max_epoch' in line or 'train_conf' in line or 'log_interval' in line:
            print(line.rstrip())

# deepspeed 패치: 모든 deepspeed 관련 import를 try/except로 감싸기
import glob
patched = []
for py_file in glob.glob('/content/CosyVoice/cosyvoice/**/*.py', recursive=True):
    with open(py_file) as f:
        lines = f.readlines()

    new_lines = []
    modified = False
    i = 0
    while i < len(lines):
        line = lines[i]
        stripped = line.strip()
        # import deepspeed 또는 from deepspeed... import ... 패턴
        if ('import deepspeed' in stripped or stripped.startswith('from deepspeed')) and 'try:' not in (lines[i-1].strip() if i > 0 else ''):
            indent = line[:len(line) - len(line.lstrip())]
            new_lines.append(f'{indent}try:\n')
            new_lines.append(f'{indent}    {stripped}\n')
            new_lines.append(f'{indent}except (ImportError, ModuleNotFoundError):\n')
            # from X import Y -> Y = None
            m = re.match(r'from\s+\S+\s+import\s+(.+)', stripped)
            if m:
                names = [n.strip() for n in m.group(1).split(',')]
                for name in names:
                    new_lines.append(f'{indent}    {name} = None\n')
            else:
                new_lines.append(f'{indent}    deepspeed = None\n')
            modified = True
        else:
            new_lines.append(line)
        i += 1

    if modified:
        with open(py_file, 'w') as f:
            f.writelines(new_lines)
        patched.append(py_file.replace('/content/CosyVoice/', ''))

if patched:
    print(f'deepspeed patched ({len(patched)} files):')
    for p in patched:
        print(f'  {p}')
else:
    print('deepspeed already patched')

train_conf:
    max_epoch: 200
    log_interval: 100
train_conf_gan:
    max_epoch: 200
    log_interval: 100
deepspeed already patched


---
## 10. Dry Run (1 Epoch)

학습이 정상 동작하는지 확인합니다.

In [34]:
# 경로가 다를 경우 사용하시는 경로에 맞춰 수정해 주세요.
!rm -rf /content/cosyvoice3_data/train/parquet
!rm -rf /content/cosyvoice3_data/dev/parquet

In [35]:
# text 파일의 내용을 그대로 instruct_text라는 이름으로 복사합니다.
!cp /content/cosyvoice3_data/train/text /content/cosyvoice3_data/train/instruct_text
!cp /content/cosyvoice3_data/dev/text /content/cosyvoice3_data/dev/instruct_text

In [39]:
# 1. 저장할 parquet 폴더를 미리 생성합니다 (핵심!)
!mkdir -p /content/cosyvoice3_data/train/parquet
!mkdir -p /content/cosyvoice3_data/dev/parquet

# 2. Train 데이터 Parquet 생성
!python /content/CosyVoice/tools/make_parquet_list.py \
    --num_utts_per_parquet 1000 \
    --src_dir /content/cosyvoice3_data/train \
    --des_dir /content/cosyvoice3_data/train/parquet

# 3. Dev(검증) 데이터 Parquet 생성
!python /content/CosyVoice/tools/make_parquet_list.py \
    --num_utts_per_parquet 1000 \
    --src_dir /content/cosyvoice3_data/dev \
    --des_dir /content/cosyvoice3_data/dev/parquet

100% 387/387 [00:00<00:00, 13108.36it/s]
100% 42/42 [00:00<00:00, 12123.10it/s]


In [41]:
# instruct_token이 없을 경우 에러를 내는 대신 text_token을 가져다 쓰도록 llm.py 코드를 자동 수정(패치)합니다.
!sed -i "s/batch\['instruct_token'\]/batch.get('instruct_token', batch['text_token'])/g" /content/CosyVoice/cosyvoice/llm/llm.py
!sed -i "s/batch\['instruct_token_len'\]/batch.get('instruct_token_len', batch['text_token_len'])/g" /content/CosyVoice/cosyvoice/llm/llm.py

In [43]:
# 파일 상단의 cosyvoice_join 함수 내부를 무조건 False로 리턴하도록 바꿉니다.
!sed -i 's/def cosyvoice_join(group_join, info_dict):/def cosyvoice_join(group_join, info_dict):\n    return False/g' /content/CosyVoice/cosyvoice/utils/train_utils.py

In [45]:
# GradScaler 객체는 만들되, enabled=False 옵션을 주어 스케일링을 하지 않고 패스하도록 만듭니다.
!sed -i 's/scaler = torch.cuda.amp.GradScaler() if args.use_amp else None/scaler = torch.cuda.amp.GradScaler(enabled=False) if args.use_amp else None/g' /content/CosyVoice/cosyvoice/bin/train.py

In [47]:
import os

# 1. 학습 유틸리티 패치: 자동 형변환(AMP)을 무조건 bfloat16으로 고정
file1 = '/content/CosyVoice/cosyvoice/utils/train_utils.py'
with open(file1, 'r') as f:
    data1 = f.read()
data1 = data1.replace('dtype=dtype', 'dtype=torch.bfloat16')
data1 = data1.replace('enabled=scaler is not None', 'enabled=True')
with open(file1, 'w') as f:
    f.write(data1)

# 2. 메인 학습 스크립트 패치: 모델 전체를 명시적으로 bfloat16으로 변환하여 메모리에 올림
file2 = '/content/CosyVoice/cosyvoice/bin/train.py'
with open(file2, 'r') as f:
    data2 = f.read()
# 코랩 환경에 따라 model = model.cuda() 형태일 수 있으므로 일괄 변환
data2 = data2.replace('model.cuda()', 'model.cuda().bfloat16()')
with open(file2, 'w') as f:
    f.write(data2)

print("✅ BFloat16 데이터 타입 강제 통일 패치 완료!")

✅ BFloat16 데이터 타입 강제 통일 패치 완료!


In [53]:
# train_utils.py 파일을 열어서 prefetch_factor 할당 로직을 PyTorch 2.x 버전에 맞게 수정합니다.
file_path = '/content/CosyVoice/cosyvoice/utils/train_utils.py'

with open(file_path, 'r') as f:
    code = f.read()

# num_workers가 0보다 클 때만 args.prefetch를 쓰고, 아니면 None을 넣도록 우회합니다.
code = code.replace(
    'prefetch_factor=args.prefetch',
    'prefetch_factor=args.prefetch if args.num_workers > 0 else None'
)

with open(file_path, 'w') as f:
    f.write(code)

print("✅ DataLoader Prefetch 에러 패치 완료!")

✅ DataLoader Prefetch 에러 패치 완료!


In [56]:
%%bash
export NCCL_P2P_DISABLE=1
export NCCL_IB_DISABLE=1
# 실시간 출력을 막는 파이썬 버퍼링 해제
export PYTHONUNBUFFERED=1

cd /content/CosyVoice/examples/libritts/cosyvoice3
. ./path.sh
export CUDA_VISIBLE_DEVICES="0"

PRETRAINED="../../../pretrained_models/Fun-CosyVoice3-0.5B"

# tail -80을 삭제하여 모든 로그가 실시간으로 보이게 뚫어놓았습니다.
torchrun --nnodes=1 --nproc_per_node=1 \
    --rdzv_id=1986 --rdzv_backend="c10d" --rdzv_endpoint="localhost:1234" \
  ../../../cosyvoice/bin/train.py \
  --train_engine torch_ddp \
  --config conf/cosyvoice3.yaml \
  --train_data /content/cosyvoice3_data/train/parquet/data.list \
  --cv_data /content/cosyvoice3_data/dev/parquet/data.list \
  --qwen_pretrain_path $PRETRAINED/CosyVoice-BlankEN \
  --onnx_path $PRETRAINED \
  --model llm \
  --checkpoint $PRETRAINED/llm.pt \
  --model_dir `pwd`/exp/dryrun/llm/torch_ddp \
  --tensorboard_dir `pwd`/tensorboard/dryrun \
  --ddp.dist_backend nccl \
  --num_workers 0 \
  --use_amp

echo ""
nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader

Process is terminated.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## 11. 본 학습 (LLM SFT)

In [57]:
import threading, time, shutil, os

stop_backup = threading.Event()

def auto_backup():
    EXP = '/content/CosyVoice/examples/libritts/cosyvoice3/exp/cosyvoice3/llm/torch_ddp'
    BAK = '/content/drive/MyDrive/cosyvoice3_backup/exp'
    while not stop_backup.is_set():
        time.sleep(300)
        if os.path.exists(EXP):
            try:
                pts = sorted([f for f in os.listdir(EXP) if f.endswith('.pt')],
                    key=lambda x: os.path.getmtime(os.path.join(EXP, x)), reverse=True)
                if pts:
                    os.makedirs(BAK, exist_ok=True)
                    dst = os.path.join(BAK, pts[0])
                    if not os.path.exists(dst):
                        shutil.copy2(os.path.join(EXP, pts[0]), dst)
                        print(f'[Backup] {pts[0]}')
            except Exception as e:
                print(f'[Backup Error] {e}')

threading.Thread(target=auto_backup, daemon=True).start()
print('Auto backup started (5min)')

Auto backup started (5min)


In [ ]:
%%bash
export NCCL_P2P_DISABLE=1
export NCCL_IB_DISABLE=1
export PYTHONUNBUFFERED=1

cd /content/CosyVoice/examples/libritts/cosyvoice3
. ./path.sh
export CUDA_VISIBLE_DEVICES="0"

# 🌟 핵심 패치: 100스텝마다 출력되던 답답한 로그를 '1스텝'마다 출력하도록 강제 수정
sed -i 's/log_interval: [0-9]*/log_interval: 1/g' conf/cosyvoice3.yaml
# (선택) 파인튜닝에 맞게 최대 에포크 수를 50으로 고정
sed -i 's/max_epoch: [0-9]*/max_epoch: 50/g' conf/cosyvoice3.yaml

PRETRAINED="../../../pretrained_models/Fun-CosyVoice3-0.5B"

torchrun --nnodes=1 --nproc_per_node=1 \
    --rdzv_id=1986 --rdzv_backend="c10d" --rdzv_endpoint="localhost:1234" \
  ../../../cosyvoice/bin/train.py \
  --train_engine torch_ddp \
  --config conf/cosyvoice3.yaml \
  --train_data /content/cosyvoice3_data/train/parquet/data.list \
  --cv_data /content/cosyvoice3_data/dev/parquet/data.list \
  --qwen_pretrain_path $PRETRAINED/CosyVoice-BlankEN \
  --onnx_path $PRETRAINED \
  --model llm \
  --checkpoint $PRETRAINED/llm.pt \
  --model_dir `pwd`/exp/cosyvoice3/llm/torch_ddp \
  --tensorboard_dir `pwd`/tensorboard/cosyvoice3/llm/torch_ddp \
  --ddp.dist_backend nccl \
  --num_workers 0 \
  --use_amp

In [61]:
%%bash
export NCCL_P2P_DISABLE=1
export NCCL_IB_DISABLE=1
export PYTHONUNBUFFERED=1

cd /content/CosyVoice/examples/libritts/cosyvoice3
. ./path.sh
export CUDA_VISIBLE_DEVICES="0"

# 🌟 핵심 패치: 100스텝마다 출력되던 답답한 로그를 '1스텝'마다 출력하도록 강제 수정
sed -i 's/log_interval: [0-9]*/log_interval: 1/g' conf/cosyvoice3.yaml
# (선택) 파인튜닝에 맞게 최대 에포크 수를 50으로 고정
sed -i 's/max_epoch: [0-9]*/max_epoch: 50/g' conf/cosyvoice3.yaml

PRETRAINED="../../../pretrained_models/Fun-CosyVoice3-0.5B"

torchrun --nnodes=1 --nproc_per_node=1 \
    --rdzv_id=1986 --rdzv_backend="c10d" --rdzv_endpoint="localhost:1234" \
  ../../../cosyvoice/bin/train.py \
  --train_engine torch_ddp \
  --config conf/cosyvoice3.yaml \
  --train_data /content/cosyvoice3_data/train/parquet/data.list \
  --cv_data /content/cosyvoice3_data/dev/parquet/data.list \
  --qwen_pretrain_path $PRETRAINED/CosyVoice-BlankEN \
  --onnx_path $PRETRAINED \
  --model llm \
  --checkpoint $PRETRAINED/llm.pt \
  --model_dir `pwd`/exp/cosyvoice3/llm/torch_ddp \
  --tensorboard_dir `pwd`/tensorboard/cosyvoice3/llm/torch_ddp \
  --ddp.dist_backend nccl \
  --num_workers 0 \
  --use_amp

[Backup] epoch_2_whole.pt
Process is terminated.


In [ ]:
stop_backup.set()

import re, matplotlib.pyplot as plt, os

train_losses, cv_losses = [], []
if os.path.exists('/content/train_log.txt'):
    with open('/content/train_log.txt') as f:
        for line in f:
            m = re.search(r'train_loss[=:\s]+([\d.]+)', line)
            if m: train_losses.append(float(m.group(1)))
            m = re.search(r'cv_loss[=:\s]+([\d.]+)', line)
            if m: cv_losses.append(float(m.group(1)))

if train_losses:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(train_losses); axes[0].set_title('Train Loss'); axes[0].grid(True)
    if cv_losses:
        axes[1].plot(cv_losses, color='orange'); axes[1].set_title('CV Loss'); axes[1].grid(True)
    plt.tight_layout(); plt.show()
    print(f'Train: {train_losses[0]:.4f} -> {train_losses[-1]:.4f}')
    if cv_losses: print(f'CV: {cv_losses[0]:.4f} -> {cv_losses[-1]:.4f} (best: {min(cv_losses):.4f})')
else:
    print('No loss found')

---
## 12. Checkpoint 평균화 + 음성 평가

In [ ]:
%%bash
cd /content/CosyVoice/examples/libritts/cosyvoice3
. ./path.sh

EXP_DIR="exp/cosyvoice3/llm/torch_ddp"
ls -lh $EXP_DIR/*.pt 2>/dev/null | tail -10

python ../../../cosyvoice/bin/average_model.py \
    --dst_model $EXP_DIR/llm.pt \
    --src_path $EXP_DIR \
    --num 5 \
    --val_best

In [ ]:
import sys, os, shutil
sys.path.insert(0, '/content/CosyVoice')
sys.path.insert(0, '/content/CosyVoice/third_party/Matcha-TTS')

PRETRAINED = '/content/CosyVoice/pretrained_models/Fun-CosyVoice3-0.5B'
EXP_DIR = '/content/CosyVoice/examples/libritts/cosyvoice3/exp/cosyvoice3/llm/torch_ddp'

llm_orig = os.path.join(PRETRAINED, 'llm.pt')
llm_bak = os.path.join(PRETRAINED, 'llm.pt.bak')
if not os.path.exists(llm_bak):
    shutil.copy2(llm_orig, llm_bak)
shutil.copy2(os.path.join(EXP_DIR, 'llm.pt'), llm_orig)
print('Fine-tuned LLM applied')

from cosyvoice.cli.cosyvoice import CosyVoice3
cosyvoice = CosyVoice3(PRETRAINED, load_trt=False)
print('Model loaded')

In [ ]:
import os, soundfile as sf
from IPython.display import Audio, display

tests = [
    ("나한텐 일상이었어", "[excited]"),
    ("미안해, 다음엔 더 잘할게", "[sad]"),
    ("오~ 꽤하잖아!", "[happy]"),
    ("한숨 돌렸다 가자고. 시간은 충분하니까", "[calm]"),
]

os.makedirs('/content/test_outputs', exist_ok=True)

for i, (text, style) in enumerate(tests):
    print(f'\n--- {style} "{text}" ---')
    for spk in ['debi', 'marlene']:
        try:
            wavs = sorted([f for f in os.listdir('/content/cosyvoice3_data/train/wavs/') if f.startswith(spk)])
            if not wavs: continue
            ref = f'/content/cosyvoice3_data/train/wavs/{wavs[0]}'
            out = None
            for r in cosyvoice.inference_instruct2(
                f'{style}{text}', 'You are a helpful assistant.<|endofprompt|>', ref, stream=False
            ):
                out = r['tts_speech'].numpy()
            if out is not None:
                path = f'/content/test_outputs/test_{i+1}_{spk}_{style.strip("[]")}.wav'
                sf.write(path, out.flatten(), 24000)
                print(f'  {spk}: OK')
                display(Audio(out.flatten(), rate=24000))
        except Exception as e:
            print(f'  {spk}: {e}')

---
## 13. 저장 (Drive + HuggingFace)

In [ ]:
import shutil, os

PRETRAINED = '/content/CosyVoice/pretrained_models/Fun-CosyVoice3-0.5B'
SAVE_DIR = '/content/drive/MyDrive/cosyvoice3_backup/final_model'
os.makedirs(SAVE_DIR, exist_ok=True)

for f in ['llm.pt', 'flow.pt', 'hift.pt', 'campplus.onnx', 'speech_tokenizer_v3.onnx', 'cosyvoice3.yaml']:
    src = os.path.join(PRETRAINED, f)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(SAVE_DIR, f))
        print(f'  {f}: {os.path.getsize(src)/1e6:.1f}MB')

for d in os.listdir(PRETRAINED):
    src = os.path.join(PRETRAINED, d)
    if os.path.isdir(src):
        dst = os.path.join(SAVE_DIR, d)
        if os.path.exists(dst): shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'  {d}/')

ref_dir = os.path.join(SAVE_DIR, 'references')
os.makedirs(ref_dir, exist_ok=True)
for spk in ['debi', 'marlene']:
    wavs = sorted([f for f in os.listdir('/content/cosyvoice3_data/train/wavs/') if f.startswith(spk)])
    if wavs:
        shutil.copy2(f'/content/cosyvoice3_data/train/wavs/{wavs[0]}', os.path.join(ref_dir, f'{spk}.wav'))

print(f'\nSaved: {SAVE_DIR}')

In [ ]:
from huggingface_hub import HfApi, login
login()

api = HfApi()
api.create_repo(repo_id='2R4mi/cosyvoice3-debi-marlene', exist_ok=True, private=True)
api.upload_folder(
    folder_path='/content/drive/MyDrive/cosyvoice3_backup/final_model',
    repo_id='2R4mi/cosyvoice3-debi-marlene',
    commit_message='CosyVoice3 fine-tuned: debi + marlene (LLM SFT)',
)
print('Uploaded')